# SQL SFT Training Loop

This notebook runs the current manifest-driven LoRA SFT loop for `Qwen/Qwen3.5-0.8B-Base`.

It uses the package code directly instead of reimplementing the trainer in notebook cells:

- manifest: `experiments/sql/qwen35_0_8b__exp001_sql_sft.json`
- seed train set: `datasets/sql/train/qwen35_0_8b_direct_sql_seed_v1.jsonl`
- adapter output: `artifacts/sql/qwen35_0_8b__exp001_sql_sft/adapter`
- optional MLflow tracking: `mlflow.db`

Run from the repo root with the training dependencies installed.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src/sqlbench_lab").exists():
            return candidate
    raise RuntimeError("Could not find SQLBench Lab repo root")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("UV_CACHE_DIR", "/tmp/uv-cache")

MANIFEST_PATH = ROOT / "experiments/sql/qwen35_0_8b__exp001_sql_sft.json"
SUMMARY_PATH = ROOT / "artifacts/sql/qwen35_0_8b__exp001_sql_sft/train_summary.json"

print(f"repo root: {ROOT}")
print(f"manifest:  {MANIFEST_PATH.relative_to(ROOT)}")

## Validate The Experiment Contract

Load the manifest, train rows, and smoke eval cases before touching the model.

In [ ]:
from sqlbench_lab.sql import load_sql_eval_cases, load_sql_sft_manifest, load_sql_train_examples

manifest = load_sql_sft_manifest(MANIFEST_PATH)
train_rows = [
    row
    for dataset_path in manifest.train_inputs.train_datasets
    for row in load_sql_train_examples(dataset_path)
]
smoke_cases = load_sql_eval_cases(manifest.eval_plan.smoke_dataset)

print(f"experiment_id: {manifest.experiment_id}")
print(f"base_model:    {manifest.student.base_model}")
print(f"train_rows:    {len(train_rows)}")
print(f"smoke_cases:   {len(smoke_cases)}")
print(f"adapter_dir:   {manifest.output_paths.adapter_dir}")

In [ ]:
first = train_rows[0]
print(first.question)
print(first.target_sql)

## Dry Run

The dry run validates the manifest and renders/tokenization inputs without loading or training the model.

In [ ]:
from sqlbench_lab.sql import run_sql_sft

dry_summary = run_sql_sft(MANIFEST_PATH, dry_run=True)
dry_summary

## Training Environment Check

This confirms the notebook kernel is using the same stack needed by the CLI training path.

In [ ]:
import datasets
import peft
import torch
import transformers

print(f"torch:        {torch.__version__}")
print(f"cuda:         {torch.cuda.is_available()}")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"datasets:     {datasets.__version__}")

## MLflow Observability

Keep MLflow disabled for quick notebook checks. Set `ENABLE_MLFLOW = True` before the training cell to log the run to the local tracking directory.

In [ ]:
ENABLE_MLFLOW = False
MLFLOW_TRACKING_URI = f"sqlite:///{ROOT / 'mlflow.db'}"
MLFLOW_EXPERIMENT = "sqlbench-lab"

print(f"mlflow enabled:      {ENABLE_MLFLOW}")
print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")

## Run One SFT Loop

This is the same call the CLI uses. It trains one epoch over the seed dataset, writes the LoRA adapter under `artifacts/`, and can log the run to MLflow.

In [ ]:
train_summary = run_sql_sft(
    MANIFEST_PATH,
    dry_run=False,
    log_mlflow=ENABLE_MLFLOW,
    mlflow_tracking_uri=MLFLOW_TRACKING_URI,
    mlflow_experiment=MLFLOW_EXPERIMENT,
)
train_summary

## Inspect Outputs

The adapter and train summary are generated artifacts and are intentionally ignored by git.

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

In [ ]:
adapter_dir = ROOT / manifest.output_paths.adapter_dir
for path in sorted(adapter_dir.iterdir()):
    print(path.relative_to(ROOT))